<a href="https://colab.research.google.com/github/Jabbiks/ML/blob/main/%D0%9B%D0%A09_%D0%90%D0%B2%D1%80%D0%B0%D0%BC%D0%B5%D0%BD%D0%BA%D0%BE_%D0%A4%D0%86%D0%A23_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Авраменко Кирило Володимирович  
**Група:** ФІТ 3-10  
**Дата:** 18.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 – Дослідіть набір даних:
print("Кількість унікальних користувачів:", ratings_df['user_id'].nunique())
print("Кількість унікальних фільмів:", ratings_df['item_id'].nunique())
print("Мінімальний рейтинг:", ratings_df['rating'].min())
print("Максимальний рейтинг:", ratings_df['rating'].max())
print(f"Середній рейтинг: {ratings_df['rating'].mean():.4f}")
print("\nПропущені значення:")
print(ratings_df.isnull().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальний рейтинг: 1.0
Максимальний рейтинг: 5.0
Середній рейтинг: 3.5299

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**


Набір даних `ml-100k` містить 100 000 записів про взаємодії користувачів із фільмами без пропущених значень у жодному зі стовпців[cite: 70]. Оцінки виставлено за шкалою від `1.0` до `5.0`[cite: 70], при цьому середній бал зміщений у бік позитивних оцінок (близько 3.53), а кількість унікальних фільмів перевищує кількість унікальних користувачів, що створює розріджену матрицю «користувач — об'єкт».

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 – Створіть train_df та test_df із ratings_df.
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [6]:
# TODO 3.2 – Підготуйте об'єкти для Surprise:
reader = Reader(
    rating_scale=data.reader.rating_scale
)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None)
)

print('Розмір train_df:', len(train_df))
print('Розмір testset:', len(testset))

Розмір train_df: 80000
Розмір testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [8]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**


* **Для SVD:** обрано кількість прихованих факторів `n_factors` (`[50, 100]`), що визначає складність латентного простору вподобань; швидкість навчання `lr_all` (`[0.005, 0.010]`) для контролю кроку градієнтного спуску та коефіцієнт регуляризації `reg_all` (`[0.02, 0.05]`), який запобігає перенавчанню на розріджених оцінках[cite: 73, 82].
* **Для KNNBasic:** задано максимальну кількість сусідів `k` (`[20, 40]`) та мінімальний поріг спільних оцінок `min_k` (`[1, 5]`) для відсіювання випадкових збігів[cite: 73, 82]. У `sim_options` перевіряються дві метрики схожості (`cosine` та кореляція `pearson`) і два підходи до фільтрації — на основі подібності об'єктів (`user_based=False`) та користувачів (`user_based=True`)[cite: 73, 82].

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
# TODO 5.1 – Підберіть гіперпараметри SVD.
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [10]:
# TODO 5.2 – Підберіть гіперпараметри KNNBasic.
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [11]:
# TODO 5.3 – Виведіть найкращий RMSE, MAE та параметри обох моделей.
print('\nНайкращий результат SVD:')
print('RMSE:', grid_search_svd.best_score['rmse'])
print('MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nНайкращий результат KNNBasic:')
print('RMSE:', grid_search_knn.best_score['rmse'])
print('MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])


Найкращий результат SVD:
RMSE: 0.9441790020144181
MAE: 0.7455089642110386
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Найкращий результат KNNBasic:
RMSE: 1.0278262445826671
MAE: 0.8154659200826685
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
| :--- | :---: | :---: | :--- |
| **SVD** | `0.9442` (`0.944179`) | `0.7455` (`0.745509`) | `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}` |
| **KNNBasic** | `1.0278` (`1.027826`) | `0.8155` (`0.815466`) | `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}` |

**Порівняння та висновки:**
За результатами перехресної перевірки модель матричної факторизації **SVD** продемонструвала помітно вищу точність прогнозування порівняно з алгоритмом **KNNBasic** за обома метриками: її помилка `CV RMSE` становить **0.9442** проти **1.0278** у KNNBasic, а середня абсолютна помилка `CV MAE` — **0.7455** проти **0.8155**[cite: 89]. Для **SVD** найкращою виявилася конфігурація з 50 прихованими факторами (`n_factors=50`), швидкістю навчання `lr_all=0.01` та коефіцієнтом регуляризації `reg_all=0.05`[cite: 89]. Для **KNNBasic** оптимальний результат забезпечила фільтрація на основі подібності користувачів (`user_based=True`) із використанням косинусної метрики (`name='cosine'`), максимальної кількості сусідів `k=40` та мінімального порогу спільних оцінок `min_k=1`[cite: 89].

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [12]:
# TODO 6.1 – Отримайте best_svd_params і best_knn_params за CV RMSE.
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 – Створіть trainset із train_data.
trainset = train_data.build_full_trainset()

# TODO 6.3 – Навчіть best_svd і best_knn.
best_svd = SVD(
    **best_svd_params,
    random_state=42
)
best_svd.fit(trainset)

best_knn = KNNBasic(
    **best_knn_params
)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [13]:
# TODO 7.1 – Отримайте svd_predictions та knn_predictions.
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 – Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
print('\nРезультати SVD на тестовій вибірці:')
svd_rmse = accuracy.rmse(svd_predictions, verbose=True)
svd_mae = accuracy.mae(svd_predictions, verbose=True)

print('\nРезультати KNNBasic на тестовій вибірці:')
knn_rmse = accuracy.rmse(knn_predictions, verbose=True)
knn_mae = accuracy.mae(knn_predictions, verbose=True)


Результати SVD на тестовій вибірці:
RMSE: 0.9192
MAE:  0.7234

Результати KNNBasic на тестовій вибірці:
RMSE: 1.0120
MAE:  0.7984


In [14]:
# TODO 7.3 – Сформуйте DataFrame results із колонками:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')

print('\nПорівняння моделей:')
display(results)

best_model_name = results.iloc[0]['Model']
print(f"Найкраща модель за RMSE: {best_model_name}")


Порівняння моделей:


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


Найкраща модель за RMSE: SVD


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
| :--- | :---: | :---: |
| **SVD** | `0.9192` (`0.919175`) | `0.7234` (`0.723355`) |
| **KNNBasic** | `1.0120` (`1.011953`) | `0.7984` (`0.798421`) |

**Порівняння з CV:**
Оцінки на незалежній тестовій вибірці не мають суттєвих розбіжностей із результатами перехресної перевірки та навіть демонструють невелике покращення якості за обома метриками: для **SVD** похибка `RMSE` знизилася з `0.9442` на CV до `0.9192` на тесті (різниця ~0.025), а для **KNNBasic** — з `1.0278` до `1.0120` (різниця ~0.016). Таке покращення пояснюється тим, що під час 3-фолдової кросвалідації кожна модель навчалася лише на 2/3 навчальних даних (~53 333 оцінки), тоді як остаточні моделі навчалися на всій навчальній вибірці (80 000 оцінок). Це підтверджує стабільну узагальнюючу здатність моделей і відсутність перенавчання.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 – Визначте найкращу за CV RMSE модель.
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    selected_model_name = 'SVD'
else:
    best_model = best_knn
    selected_model_name = 'KNNBasic'

# TODO 8.2 – Перевірте, що користувач присутній у trainset.
try:
    inner_uid = trainset.to_inner_uid(user_id)
    print(f"Для рекомендацій використовується модель: {selected_model_name}")

    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    print(f"Кількість оцінених фільмів користувачем {user_id}: {len(rated_inner_iids)}")

    # TODO 8.3 – Знайдіть неоцінені ним фільми.
    unrated_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
        if inner_iid not in rated_inner_iids
    ]

    # TODO 8.4 – Спрогнозуйте оцінки та покажіть топ-10.
    user_predictions = [
        (raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]

    recommendations_df = pd.DataFrame(
        user_predictions,
        columns=['item_id', 'predicted_rating']
    ).sort_values(by='predicted_rating', ascending=False).head(10).reset_index(drop=True)

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    display(recommendations_df)

except ValueError:
    print(
        f"Користувача з ID {user_id} "
        "немає в навчальній вибірці."
    )

Для рекомендацій використовується модель: SVD
Кількість оцінених фільмів користувачем 196: 29

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

**Висновок:**


За результатами перехресної перевірки (CV) на навчальній вибірці (80 000 записів) найкращою моделлю обрано **SVD**, оскільки вона показала нижчу похибку прогнозування (`CV RMSE = 0.9442`, `CV MAE = 0.7455`) порівняно з **KNNBasic** (`CV RMSE = 1.0278`, `CV MAE = 0.8155`). Оптимальними гіперпараметрами для SVD виявилися `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}`, тоді як для KNNBasic найкращою конфігурацією стала `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}`. На незалежній тестовій вибірці (20 000 записів) модель SVD підтвердила свою перевагу з підсумковими метриками `Test RMSE = 0.9192` та `Test MAE = 0.7234` проти `Test RMSE = 1.0120` та `Test MAE = 0.7984` у KNNBasic. Різниця між показниками кросвалідації та тестування є незначною (тестові помилки трохи нижчі завдяки донавчанню на повному `trainset`), що свідчить про високу стабільність та відсутність перенавчання. Використовуючи модель SVD, для користувача `196` (який оцінив 29 фільмів у навчальному наборі) було сформовано персоналізований топ-10 рекомендацій із прогнозованими рейтингами в діапазоні від `4.38` до `4.49`. Головним обмеженням побудованої системи колаборативної фільтрації є проблема «холодного старту» (cold start): для нових користувачів або нових фільмів, відсутніх у `trainset`, модель не має історії взаємодій і змушена повертати лише глобальний середній рейтинг.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**



In [16]:
import numpy as np
import pandas as pd

# 1. Побудова матриці взаємодій "користувач – фільм" на навчальних даних
ratings_matrix = train_df.pivot(index='user_id', columns='item_id', values='rating')
R = ratings_matrix.values

# Словники відповідності ID користувачів/фільмів та індексів матриці
user_to_idx = {uid: idx for idx, uid in enumerate(ratings_matrix.index)}
item_to_idx = {iid: idx for idx, iid in enumerate(ratings_matrix.columns)}
idx_to_item = {idx: iid for iid, idx in item_to_idx.items()}

# 2. Обчислення глобального середнього та середніх оцінок кожного користувача
global_mean = float(np.nanmean(R))
user_means = np.nanmean(R, axis=1)

# Центрування оцінок (віднімаємо середній бал користувача, пропуски заповнюємо 0)
R_centered = np.where(np.isnan(R), 0.0, R - user_means[:, np.newaxis])

# 3. Обчислення матриці косинусної схожості між користувачами
norms = np.linalg.norm(R_centered, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
R_normalized = R_centered / norms
user_similarity = R_normalized @ R_normalized.T


# 4. Функція прогнозування рейтингу для пари (user_id, item_id)
def custom_predict(uid, iid, k=30):
    u_known = uid in user_to_idx
    i_known = iid in item_to_idx

    # Обробка випадків "холодного старту"
    if not u_known and not i_known:
        return global_mean
    if not u_known:
        return float(np.nanmean(R[:, item_to_idx[iid]]))
    if not i_known:
        return float(user_means[user_to_idx[uid]])

    u_idx = user_to_idx[uid]
    i_idx = item_to_idx[iid]

    # Пошук користувачів, які вже оцінили цей фільм
    rated_users = np.where(~np.isnan(R[:, i_idx]))[0]
    if len(rated_users) == 0:
        return float(user_means[u_idx])

    # Відбір найближчих сусідів із позитивною схожістю
    sims = user_similarity[u_idx, rated_users]
    pos_mask = sims > 0
    if not np.any(pos_mask):
        return float(user_means[u_idx])

    sims = sims[pos_mask]
    neighbors = rated_users[pos_mask]

    if len(sims) > k:
        top_k_idx = np.argsort(sims)[-k:]
        sims = sims[top_k_idx]
        neighbors = neighbors[top_k_idx]

    # Прогноз як середній рейтинг користувача + зважена поправка сусідів
    neighbor_deviations = R_centered[neighbors, i_idx]
    pred = user_means[u_idx] + np.dot(sims, neighbor_deviations) / np.sum(sims)

    return float(np.clip(pred, 1.0, 5.0))


# 5. Оцінка якості власного алгоритму на тестовій вибірці (RMSE та MAE)
y_true = test_df['rating'].values
y_pred = np.array([
    custom_predict(row.user_id, row.item_id, k=30)
    for row in test_df.itertuples(index=False)
])

custom_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
custom_mae = np.mean(np.abs(y_true - y_pred))

print("Результати власного алгоритму (User-Based CF з центруванням):")
print(f"Test RMSE: {custom_rmse:.6f}")
print(f"Test MAE:  {custom_mae:.6f}")

# 6. Генерація Топ-10 рекомендацій для користувача '196'
target_user = '196'
if target_user in user_to_idx:
    u_idx = user_to_idx[target_user]
    unrated_item_indices = np.where(np.isnan(R[u_idx]))[0]

    custom_recs = [
        (idx_to_item[i_idx], custom_predict(target_user, idx_to_item[i_idx], k=30))
        for i_idx in unrated_item_indices
    ]

    custom_recs_df = (
        pd.DataFrame(custom_recs, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f"\nТоп-10 рекомендацій власного алгоритму для користувача {target_user}:")
    display(custom_recs_df)

Результати власного алгоритму (User-Based CF з центруванням):
Test RMSE: 0.933713
Test MAE:  0.727980

Топ-10 рекомендацій власного алгоритму для користувача 196:


,item_id,predicted_rating
0,1628,5.000000
1,1629,5.000000
2,1642,5.000000
3,1500,5.000000
4,1554,5.000000
5,1306,5.000000
6,1103,5.000000
7,1398,4.980296
8,1467,4.957341
9,1653,4.871921
